In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/industrial-machine-condition-classification-mtech/sample_submission.csv
/kaggle/input/competitions/industrial-machine-condition-classification-mtech/student_dataset.csv
/kaggle/input/competitions/industrial-machine-condition-classification-mtech/test.csv


In [2]:

# ============================================================
# MACHINE CONDITION CLASSIFICATION - EXTRA TREES
# Complete Kaggle Notebook
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.base import clone

warnings.filterwarnings("ignore")

SEED = 42
TARGET = "condition"
ID = "unit_id"

# ============================================================
# 1. LOAD DATA
# ============================================================

INPUT_DIR = "/kaggle/input/competitions/industrial-machine-condition-classification-mtech/"

print("Available input folders:", os.listdir(INPUT_DIR))

def find_csv(candidates):
    for root, dirs, files in os.walk(INPUT_DIR):
        for filename in candidates:
            if filename in files:
                return os.path.join(root, filename)

    raise FileNotFoundError(
        f"Could not find any of these files: {candidates}. "
        "Check that your competition dataset is attached."
    )

TRAIN_PATH = find_csv(["student_dataset.csv", "student.csv"])
TEST_PATH = find_csv(["test.csv"])

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("\nTraining shape:", train.shape)
print("Test shape:", test.shape)

# ============================================================
# 2. DEFINE FEATURES
# ============================================================

NUM = [
    "ambient_temp",
    "process_temp",
    "coolant_flow",
    "shaft_speed",
    "drive_torque",
    "power_load",
    "component_wear",
    "thermal_stress",
    "mechanical_load",
    "vibration_index"
]

CAT = ["unit_type"]

required_train = set(NUM + CAT + [ID, TARGET])
required_test = set(NUM + CAT + [ID])

if not required_train.issubset(train.columns):
    raise ValueError(
        f"Missing training columns: "
        f"{required_train - set(train.columns)}"
    )

if not required_test.issubset(test.columns):
    raise ValueError(
        f"Missing test columns: "
        f"{required_test - set(test.columns)}"
    )

X = train.drop(columns=[ID, TARGET]).copy()
y = train[TARGET].copy()
X_test = test.drop(columns=[ID]).copy()

if y.isna().any():
    raise ValueError("Missing values found in the target column.")

print("\nTarget distribution:")
print(y.value_counts().sort_index())

# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

def add_features(df):
    df = df.copy()

    df["temp_diff"] = (
        df["process_temp"] - df["ambient_temp"]
    )

    df["pw"] = (
        df["drive_torque"] * df["shaft_speed"]
    )

    df["wear_torque"] = (
        df["component_wear"] * df["drive_torque"]
    )

    df["cool_ratio"] = (
        df["temp_diff"] / (df["coolant_flow"] + 1e-6)
    )

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    return df

NUM_FE = NUM + [
    "temp_diff",
    "pw",
    "wear_torque",
    "cool_ratio"
]

# ============================================================
# 4. PREPROCESSING PIPELINE
# ============================================================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, NUM_FE),
    ("cat", categorical_pipeline, CAT)
])

# ============================================================
# 5. EXTRA TREES CLASSIFIER
# ============================================================

extra_trees = ExtraTreesClassifier(
    n_estimators=200,
    random_state=SEED,
    n_jobs=1
)

model_pipeline = Pipeline([
    ("fe", FunctionTransformer(add_features)),
    ("prep", preprocessor),
    ("clf", extra_trees)
])

# ============================================================
# 6. CROSS-VALIDATION AND HYPERPARAMETER TUNING
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

param_grid = {
    "clf__n_estimators": [200, 400],
    "clf__max_depth": [None, 10, 20],
    "clf__min_samples_leaf": [1, 3, 5],
    "clf__min_samples_split": [2, 5, 10],
    "clf__max_features": ["sqrt", 1.0]
}

grid = GridSearchCV(
    estimator=model_pipeline,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    refit=True,
    return_train_score=True
)

print("\nStarting Extra Trees hyperparameter tuning...")
grid.fit(X, y)

best_model = grid.best_estimator_

print("\n" + "=" * 50)
print("EXTRA TREES RESULTS")
print("=" * 50)

print("\nBest CV accuracy:", round(grid.best_score_, 5))
print("Best parameters:", grid.best_params_)

best_index = grid.best_index_

print(
    "Mean training accuracy:",
    round(grid.cv_results_["mean_train_score"][best_index], 5)
)

# ============================================================
# 7. OUT-OF-FOLD EVALUATION
# ============================================================

oof_predictions = np.empty(len(X), dtype=y.dtype)

for fold, (train_idx, valid_idx) in enumerate(
    cv.split(X, y), start=1
):
    fold_model = clone(best_model)

    fold_model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    oof_predictions[valid_idx] = fold_model.predict(
        X.iloc[valid_idx]
    )

    fold_accuracy = accuracy_score(
        y.iloc[valid_idx],
        oof_predictions[valid_idx]
    )

    print(f"Fold {fold} accuracy: {fold_accuracy:.5f}")

print(
    "\nOverall out-of-fold accuracy:",
    round(accuracy_score(y, oof_predictions), 5)
)

print("\nClassification report:")

print(classification_report(
    y,
    oof_predictions,
    labels=[0, 1, 2],
    target_names=[
        "Normal",
        "Thermal Failure",
        "Mechanical Failure"
    ],
    zero_division=0
))

# ============================================================
# 8. PREDICT TEST DATA
# ============================================================

# GridSearchCV has refitted the best model on all training rows.
test_predictions = best_model.predict(X_test)

# ============================================================
# 9. CREATE KAGGLE SUBMISSION FILE
# ============================================================

submission = pd.DataFrame({
    "unit_id": test[ID].to_numpy(),
    "condition": test_predictions.astype(int)
})

# Validate required submission format
assert list(submission.columns) == ["unit_id", "condition"]
assert len(submission) == len(test)
assert submission["unit_id"].equals(test[ID].reset_index(drop=True))
assert submission["condition"].isin([0, 1, 2]).all()

OUTPUT_PATH = "/kaggle/working/submission_extra_trees.csv"

submission.to_csv(OUTPUT_PATH, index=False)

print("\n" + "=" * 50)
print("SUBMISSION FILE CREATED")
print("=" * 50)

print("File:", OUTPUT_PATH)
print("Shape:", submission.shape)

print("\nFirst 10 predictions:")
print(submission.head(10))

print("\nPredicted condition counts:")
print(submission["condition"].value_counts().sort_index())

print("\nCSV preview:")
print(submission.to_csv(index=False)[:500])

print("\nReady to submit to Kaggle!")


Available input folders: ['sample_submission.csv', 'student_dataset.csv', 'test.csv']

Training shape: (16000, 13)
Test shape: (4000, 12)

Target distribution:
condition
0    8000
1    4000
2    4000
Name: count, dtype: int64

Starting Extra Trees hyperparameter tuning...
Fitting 5 folds for each of 108 candidates, totalling 540 fits
[CV] END clf__max_depth=None, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__min_samples_split=2, clf__n_estimators=200; total time=   6.1s
[CV] END clf__max_depth=None, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__min_samples_split=2, clf__n_estimators=400; total time=  11.6s
[CV] END clf__max_depth=None, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__min_samples_split=5, clf__n_estimators=200; total time=   5.2s
[CV] END clf__max_depth=None, clf__max_features=sqrt, clf__min_samples_leaf=1, clf__min_samples_split=5, clf__n_estimators=200; total time=   5.3s
[CV] END clf__max_depth=None, clf__max_features=sqrt, clf__min_samples_leaf=